Teste

In [ ]:
# ===== IMPORTS =====
import sys
import os
sys.path.append(os.path.abspath("../"))

import cv2
import time
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

from src.core.detector import Detector
from src.core.processor import DetectionProcessor
from src.core.visualizer import Visualizer
from src.core.pipeline import VisionPipeline


# ===== INIT =====
detector = Detector()

processor = DetectionProcessor(
    target_classes=["cell phone"],
    ignore_classes=["person"]
)

visualizer = Visualizer()
pipeline = VisionPipeline(detector, processor, visualizer)

cap = cv2.VideoCapture(0)

os.makedirs("outputs", exist_ok=True)

last_detected = []


# ===== LOOP CONTROLADO =====
try:
    for _ in range(10000):  # limite grande, evita loop infinito travado

        ret, frame = cap.read()
        if not ret:
            break

        processed, detections, targets = pipeline.run(frame)

        current_detected = [d["label"] for d in targets]

        if current_detected and current_detected != last_detected:
            filename = f"outputs/detect_{int(time.time())}.jpg"
            cv2.imwrite(filename, processed)

            print(f"📸 Novo detectado: {current_detected} → {filename}")

            last_detected = current_detected

        # ===== VISUAL =====
        raw_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        proc_rgb = cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)

        clear_output(wait=True)

        fig, ax = plt.subplots(1, 2, figsize=(10,5))

        ax[0].imshow(raw_rgb)
        ax[0].set_title("Camera Raw")
        ax[0].axis("off")

        ax[1].imshow(proc_rgb)
        ax[1].set_title("Detection")
        ax[1].axis("off")

        display(fig)

        time.sleep(0.03)

except KeyboardInterrupt:
    print("🛑 Interrompido pelo usuário")

finally:
    cap.release()
    print("✅ Câmera liberada")